# 08 — Extração de evidências: piloto com revisão humana

Compara base e adaptador em **seis famílias de validação**, duas por janela, escolhidas por hash antes das gerações. Produz 12 respostas nos seis campos do protocolo. Esta etapa qualifica a extração antes de ampliar às 200 famílias. Não treina e não estima F1 sem referência humana. Use GPU.

Atualize notebook, script, perfil e SHA256SUMS.txt. Preserve os arquivos anteriores e o checkpoint do 05. O prompt/schema congelados do 03 continuam em uso.


## 1. Montar Drive


In [ ]:
from pathlib import Path
from google.colab import drive
import os

os.environ['HF_HOME'] = '/content/hf_cache_llm_eidt_br'
os.environ['USE_TF'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')
PASTA_DRIVE = '/content/drive/MyDrive/Replicação LLM EIDT IC' # @param {type:'string'}

raiz = Path(PASTA_DRIVE).expanduser().resolve(strict=True)
ponto_de_montagem = Path('/content/drive').resolve(strict=True)
if raiz == ponto_de_montagem or not raiz.is_relative_to(ponto_de_montagem) or not raiz.is_dir():
    raise ValueError('Selecione a pasta do projeto dentro do Drive montado.')

def caminho_no_projeto(relativo):
    parte = Path(relativo)
    if not relativo or parte.is_absolute() or '..' in parte.parts:
        raise ValueError(f'O caminho deve ser relativo ao projeto: {relativo!r}')
    alvo = (raiz / parte).resolve()
    if alvo == raiz or not alvo.is_relative_to(raiz):
        raise ValueError('O caminho informado sai da pasta do projeto.')
    return alvo

perfil_relativo = 'config/perfil_extracao_validacao6.json'
script_relativo = 'scripts/extrair_evidencias.py'
print('Pasta selecionada:', raiz)
print('Extração: 6 famílias de validação, base e adaptado, 12 respostas.')


## 2. Conferir arquivos e checkpoint
Apenas as seis validações selecionadas entram no modelo. Nenhum CPC é incluído no prompt.


In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
from importlib import metadata
import json, re, subprocess, sys, tempfile

FIXOS = {'config/perfil_extracao_validacao6.json': '394ce3431265b6e057228a91ee3029a152390cbf9e600c71ef875fc2b91ab0e0', 'scripts/extrair_evidencias.py': '5c029ebded89a25030e9e9198b48605c937356a367793f69817ca5bd15a57b75'}
assinaturas = {}
for linha in (raiz / 'SHA256SUMS.txt').read_text(encoding='utf-8-sig').splitlines():
    if linha.strip():
        digest, nome = linha.split('  ', 1)
        if nome in assinaturas:
            raise ValueError('Entrada duplicada no índice: ' + nome)
        assinaturas[nome] = digest
def conferir(nome, esperado):
    arquivo = caminho_no_projeto(nome)
    if not arquivo.is_file():
        raise FileNotFoundError('Copie para o Drive: ' + nome)
    if sha256(arquivo.read_bytes()).hexdigest() != esperado:
        raise ValueError('Arquivo de outra versão: ' + nome)
for nome, esperado in FIXOS.items():
    conferir(nome, esperado)
    if assinaturas.get(nome) != esperado:
        raise ValueError('Atualize também o SHA256SUMS.txt.')
for nome in ('requirements.txt', 'scripts/avaliar_cpc_geracao.py', 'scripts/treinar_qlora_cpc.py', 'scripts/testar_qlora_gpu.py'):
    conferir(nome, assinaturas.get(nome))
perfil = json.loads((raiz / perfil_relativo).read_bytes())
verificacao = subprocess.run([sys.executable, '-u', str(raiz / script_relativo), '--raiz', str(raiz),
    '--perfil', perfil_relativo, '--validar_apenas'], text=True, capture_output=True)
if verificacao.returncode:
    print(verificacao.stderr[-6000:])
    raise RuntimeError('Confira os arquivos e o checkpoint antes de continuar.')
print(verificacao.stdout)
pasta_relatorios = caminho_no_projeto(perfil['resultados'])
pasta_relatorios.mkdir(parents=True, exist_ok=True)
inicio_notebook = datetime.now(timezone.utc)
relatorio_preparacao = pasta_relatorios / f'preparacao_extracao_{inicio_notebook:%Y%m%dT%H%M%S%fZ}.json'
preparacao = {'estado': 'checkpoint_conferido', 'perfil': perfil['id'], 'particao': 'validacao', 'hashes': FIXOS}
def salvar_preparacao():
    preparacao['atualizado_utc'] = datetime.now(timezone.utc).isoformat()
    tmp = relatorio_preparacao.with_suffix('.tmp')
    tmp.write_text(json.dumps(preparacao, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(relatorio_preparacao)
salvar_preparacao()
arquivo_requisitos = raiz / 'requirements.txt'
print('Preparação:', relatorio_preparacao)


## 3. Instalar dependências
Preserva PyTorch e NumPy do Colab. GPU e modelos são usados em processos separados.


In [ ]:
if preparacao.get('estado') != 'checkpoint_conferido':
    raise RuntimeError('Conclua a conferência do checkpoint antes de instalar dependências.')
try:
    versoes_protegidas = {nome: metadata.version(nome) for nome in ("torch", "numpy")}
    preparacao["versoes_protegidas"] = versoes_protegidas
    desejadas = {}
    for linha in arquivo_requisitos.read_text(encoding="utf-8-sig").splitlines():
        linha = linha.split("#", 1)[0].strip()
        if linha:
            if not re.fullmatch(r"[A-Za-z0-9_.-]+==[^\s]+", linha):
                raise ValueError("Os requisitos devem usar versões exatas no formato pacote==versão.")
            nome, versao = linha.split("==", 1)
            desejadas[nome] = versao
    divergentes = []
    for nome, esperada in desejadas.items():
        try:
            atual = metadata.version(nome)
        except metadata.PackageNotFoundError:
            atual = None
        if atual != esperada:
            divergentes.append(nome)
    preparacao["pacotes_que_requeriam_instalacao"] = divergentes
    if divergentes:
        with tempfile.TemporaryDirectory(dir="/content") as pasta_temporaria:
            restricoes = Path(pasta_temporaria) / "base_colab.txt"
            plano_pip = Path(pasta_temporaria) / "operacoes_pip.json"
            restricoes.write_text(
                "".join(f"{nome}=={versao}\n" for nome, versao in versoes_protegidas.items()),
                encoding="utf-8",
            )
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                 "--progress-bar", "off", "-c", str(restricoes), "-r", str(arquivo_requisitos),
                 "--report", str(plano_pip)],
                check=True,
            )
            operacoes = json.loads(plano_pip.read_text(encoding="utf-8"))["install"]
            preparacao["pacotes_alterados"] = sorted(item["metadata"]["name"] for item in operacoes)
    else:
        preparacao["pacotes_alterados"] = []
    for nome, anterior in versoes_protegidas.items():
        if metadata.version(nome) != anterior:
            raise RuntimeError(f"A versão protegida de {nome} mudou. Interrompa e confira o ambiente.")
    preparacao["versoes_requisitos"] = {nome: metadata.version(nome) for nome in desejadas}
    if preparacao["versoes_requisitos"] != desejadas:
        raise RuntimeError("Alguma dependência ainda diverge da versão fixada.")
    preparacao["estado"] = "dependencias_conferidas"
    print("Dependências conferidas. PyTorch e NumPy nativos preservados.")
except Exception as erro:
    preparacao["estado"] = "falha_na_instalacao"
    preparacao["tipo_erro"] = type(erro).__name__
    raise
finally:
    salvar_preparacao()

## 4. Conferir recursos
Usa Qwen3-8B NF4/FP16. O primeiro carregamento pode baixar aproximadamente 16 GB de pesos no armazenamento temporário. O contexto de extração é maior que o do CPC; a viabilidade será medida nesta execução.


In [ ]:
if preparacao.get('estado') != 'dependencias_conferidas':
    raise RuntimeError('Conclua a instalação e conferência das dependências antes de continuar.')
codigo_recursos = r"""
import json, os, platform, shutil
from pathlib import Path
import torch
ram = {}
for linha in Path('/proc/meminfo').read_text().splitlines():
    chave, _, valor = linha.partition(':')
    if chave in ('MemTotal', 'MemAvailable'):
        ram[chave] = round(int(valor.split()[0]) / 1024**2, 2)
saida = {'python': platform.python_version(), 'torch': str(torch.__version__),
         'cuda_pytorch': torch.version.cuda, 'cuda_disponivel': torch.cuda.is_available(),
         'disco_livre_gib': round(shutil.disk_usage('/content').free / 1024**3, 2), 'ram_gib': ram}
if saida['cuda_disponivel']:
    livre, total = torch.cuda.mem_get_info(0)
    saida.update(gpu=torch.cuda.get_device_name(0), capacidade_cuda=list(torch.cuda.get_device_capability(0)),
                 vram_livre_gib=round(livre/1024**3, 3), vram_total_gib=round(total/1024**3, 3))
print(json.dumps(saida))
"""
try:
    verificador = subprocess.run([sys.executable, '-c', codigo_recursos], capture_output=True, text=True, check=True)
    recursos = json.loads(verificador.stdout.strip().splitlines()[-1])
    snapshot = (Path(os.environ['HF_HOME']) / 'hub' / 'models--Qwen--Qwen3-8B' /
                'snapshots' / perfil['modelo']['revisao'])
    indice_pesos = snapshot / 'model.safetensors.index.json'
    cache_completo = False
    if indice_pesos.is_file():
        fragmentos = set(json.loads(indice_pesos.read_text())['weight_map'].values())
        cache_completo = bool(fragmentos) and all(
            Path(nome).name == nome and (snapshot / nome).is_file() and (snapshot / nome).stat().st_size > 0
            for nome in fragmentos
        )
    recursos['cache_pesos_completo'] = cache_completo
    recursos['disco_minimo_gib'] = 4 if cache_completo else 22
    preparacao['recursos'] = recursos
    print(json.dumps(recursos, ensure_ascii=False, indent=2))
    if not recursos['cuda_disponivel']:
        raise RuntimeError('Selecione GPU no tipo de ambiente de execução do Colab.')
    if recursos['capacidade_cuda'][0] < 6:
        raise RuntimeError('A GPU não atende à capacidade CUDA mínima para este perfil NF4.')
    if recursos['disco_livre_gib'] < recursos['disco_minimo_gib']:
        raise RuntimeError('O armazenamento local está abaixo da reserva necessária. Revise o espaço antes da avaliação.')
    preparacao['estado'] = 'pronto_para_avaliacao'
    print('Ambiente preparado. A próxima célula inicia as 12 gerações de extração.')
except Exception as erro:
    preparacao['estado'] = 'recursos_requerem_atencao'
    preparacao['tipo_erro'] = type(erro).__name__
    if isinstance(erro, subprocess.CalledProcessError):
        print((erro.stderr or '')[-4000:])
    raise
finally:
    salvar_preparacao()

## 5. Gerar e salvar as 12 respostas
Uma família por vez, até 2.048 tokens de entrada e 2.048 de saída, sem recorte. O programa salva cada resposta no Drive, inclusive JSON inválido ou truncado. Na retomada, reutiliza todas as respostas concluídas e gera somente as faltantes. Erro de GPU interrompe a execução e exige análise, sem reduzir limites automaticamente.

Não rode duas sessões simultâneas. Se a sessão cair, execute novamente o notebook com o mesmo perfil. A trava residual só deve ser removida após encerrar a sessão anterior.


In [ ]:
if preparacao.get('estado') != 'pronto_para_avaliacao':
    raise RuntimeError('Conclua as conferências anteriores.')
pasta_execucao = pasta_relatorios / ('extracao_' + sha256((raiz / perfil_relativo).read_bytes()).hexdigest()[:16])
# Marque True SOMENTE após confirmar que a sessão anterior foi encerrada.
# Uma queda abrupta pode deixar a trava. Respostas salvas serão preservadas.
LIBERAR_TRAVA_DE_SESSAO_ENCERRADA = False # @param {type:'boolean'}
trava = pasta_execucao / 'em_execucao.lock'
if trava.exists():
    if not LIBERAR_TRAVA_DE_SESSAO_ENCERRADA:
        raise RuntimeError('Há uma trava de execução. Confirme que nenhum outro Colab está rodando este notebook. Se a sessão caiu, marque a opção acima e execute esta célula.')
    trava.unlink()
preparacao['estado'] = 'extracao_em_execucao'
preparacao['pasta_execucao'] = str(pasta_execucao)
salvar_preparacao()
print('Resultados e retomada:', pasta_execucao, flush=True)
processo = None
try:
    processo = subprocess.run([sys.executable, '-u', str(raiz / script_relativo), '--raiz', str(raiz), '--perfil', perfil_relativo], check=False)
finally:
    caminho_relatorio = pasta_execucao / 'relatorio.json'
    if caminho_relatorio.exists():
        relatorio = json.loads(caminho_relatorio.read_bytes())
        print('RESUMO DA EXTRAÇÃO — copie este bloco')
        print(json.dumps({k: relatorio.get(k) for k in ('estado','geracoes_concluidas','metricas_automaticas','retomadas_do_drive','memoria_final','erro')}, ensure_ascii=False, indent=2))
        print('Relatório:', caminho_relatorio)
    preparacao['estado'] = 'extracao_concluida' if processo and processo.returncode == 0 else 'extracao_requer_atencao'
    salvar_preparacao()
if not processo or processo.returncode:
    raise RuntimeError('Preserve o relatório. A retomada reutiliza as respostas já salvas; não altere perfil nem prompt.')
print('Geração encerrada. Revisão humana pendente. Preserve a pasta completa.')


## 6. Enviar resultados e organizar revisão
Envie o resumo, `relatorio.json` e `extracoes.jsonl` da pasta mostrada. A pasta `respostas/` preserva cada retorno bruto.

Antes de ver as previsões, o anotador deve trabalhar em uma cópia de `anotacao_independente.jsonl`, preencher os seis campos conforme o guia e registrar sua identificação. `null` significa pendente, `[]` ausência após revisão. Depois, revisar uma cópia de `revisao_cega_modelos.csv`; não enviar a chave A/B ao anotador. Não editar arquivos de geração originais. Consulte `docs/EXTRACAO_E_REVISAO_08.md`.

Validade do JSON e dos trechos não comprova correção técnica. Não expandir para as 200 nem escolher o modelo só pela taxa de formato válido.
